In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import random
#### xy pixel size = 0.1893014, z pixel size = 0.7996567
XY_UM = 0.1893014
Z_UM = 0.7996567

output_dir = Path(r"c:\Users\taylorhearn\git_repos\image_quantification\New_Spacefish")

# Read in the Data (Skip this, already loaded)
- You can now skip this and just read in the data directly from the csv 

In [7]:
# def source_root_name(folder: Path) -> str:
#     """Name of the top-level root in folders_to_read that contains this folder."""
#     rp = folder.resolve()
#     for root in folders_to_read:
#         rr = root.resolve()
#         if rp == rr or rr in rp.parents:
#             return root.name
#     return "UNKNOWN"

# def mask_extent(seg_path: Path):
#     """Read only the array shape (no pixel data) -> (Xmax, Ymax, Zmax)."""
#     shape = tifffile.TiffFile(str(seg_path)).series[0].shape
#     zmax, ymax, xmax = tuple(s for s in shape if s > 1)  # drop singleton axes
#     return int(xmax), int(ymax), int(zmax)


# directory = Path(r"z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results")
# folders = []
# for folder in directory.iterdir():
#     if folder.is_dir():
#         folders.append(folder)
# folders_to_read = [f for f in folders if "noamp" not in f.name and "noim" not in f.name]

# all_tables = []

# for folder in folders_to_read:
#     print(folder)
#     count_csv = next(iter(folder.glob("*count_table*.csv")), None)
#     seg_path = next(iter(folder.glob("*seg_mask.tif")), None)
#     xyz_path = next(iter(folder.glob("*NucleiLocation*.csv")), None)

#     if seg_path is None:
#         print(f"[SKIP - no seg_mask] {folder}")
#         continue
#     if not xyz_path.exists():
#         print(f"[SKIP - no nucleus_xyz.csv] {folder}")
#         continue
#     if not count_csv.exists():
#         print(f"[SKIP - no count_table.csv] {folder}")
#         continue

#     df = pd.read_csv(count_csv)

#     # Centroids from nucleus_xyz.csv (fast). NOTE: validated x<->y swap vs seg_mask.
#     xyz = pd.read_csv(xyz_path).rename(columns={
#         "ID": "nucleus",
#         "x": "nucleus_centroid_y",   # csv 'x' is image row (Y)
#         "y": "nucleus_centroid_x",   # csv 'y' is image column (X)
#         "z": "nucleus_centroid_z",
#     })[["nucleus", "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_centroid_z"]]
#     df = df.merge(xyz, on="nucleus", how="left")

#     # Extent from mask header only (no full load).
#     xmax, ymax, zmax = mask_extent(seg_path)
#     df["Xmax"], df["Ymax"], df["Zmax"] = xmax, ymax, zmax

#     # Provenance: which root folder this image came from.
#     df.insert(0, "source_folder", source_root_name(folder))

#     out_name = f"{folder.name}_count_table_xyz.csv"
#     df.to_csv(output_dir / out_name, index=False)
#     all_tables.append(df)
#     print(f"[OK] {folder.name} [{df['source_folder'].iloc[0]}]: "
#           f"{len(df)} nuclei, extent (X,Y,Z)=({xmax},{ymax},{zmax})")

# # Concatenate every nucleus from every image into one table.
# if all_tables:
#     all_data = pd.concat(all_tables, ignore_index=True)
#     all_data.to_csv(output_dir / "all_nuclei.csv", index=False)
#     print(f"\nall_data table: {all_data.shape[0]} nuclei x {all_data.shape[1]} columns")
#     print(f"Saved -> {output_dir / 'all_nuclei.csv'}")
# else:
#     print("\nNo folders produced output.")


In [84]:
def define_cell_type(ne, nf):
    """Classify a cell along the endothelial<->fibroblast axis with a confidence tag.
    Confident = at least 2 of its own markers and none of the other lineage's.
    Tentative compares the FRACTION of each panel detected (ne/6 vs nf/3), not raw
    counts, so the larger endothelial panel doesn't bias the tie-break.
    Returns (cell_type, confidence); confidence is <NA> for Unknown cells."""
    if ne >= 2 and nf == 0:
        return "Endothelial", "Confident"
    if nf >= 2 and ne == 0:
        return "Fibroblast", "Confident"
    endo_frac = ne / len(endo_genes)
    fibro_frac = nf / len(fibro_genes)
    if endo_frac > fibro_frac:
        return "Endothelial", "Tentative"
    if fibro_frac > endo_frac:
        return "Fibroblast", "Tentative"
    return "Unknown", pd.NA               # tie (incl. no markers at all)

In [92]:
all_data = pd.read_csv(output_dir / "all_nuclei.csv")

In [93]:
endo_genes = ["CDH5", "PECAM1", "VWF", "KDR", "FLT1", "PDGFB"]
fibro_genes = ["PDGFRB", "COL1A1", "COL1A2"]

endo_on = (all_data[endo_genes].fillna(0) > 0).sum(axis=1)
fibro_on = (all_data[fibro_genes].fillna(0) > 0).sum(axis=1)
all_data["n_endo_markers"] = endo_on
all_data["n_fibro_markers"] = fibro_on
all_data["endothelial_score"] = endo_on / len(endo_genes)
all_data["fibroblast_score"] = fibro_on / len(fibro_genes)

all_data[["cell_type", "confident"]] = pd.DataFrame(
    [define_cell_type(ne, nf) for ne, nf in zip(endo_on, fibro_on)],
    index=all_data.index,
)

# has_reads: True if the cell has any transcript across every gene panel (FP channels excluded)
non_gene_cols = {
    "source_folder", "name", "nucleus",
    "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_centroid_z",
    "Xmax", "Ymax", "Zmax", "n_endo_markers", "n_fibro_markers",
    "endothelial_score", "fibroblast_score", "cell_type", "confident", "has_reads",
}
gene_cols_all = [c for c in all_data.columns if c not in non_gene_cols and not c.startswith("FP")]
all_data["has_reads"] = (all_data[gene_cols_all].fillna(0) > 0).sum(axis=1) > 0

# Cells with no reads override the marker-based label; confidence is undefined for them

all_data.loc[~all_data["has_reads"], "cell_type"] = "Zero Reads"
all_data.loc[~all_data["has_reads"], "confident"] = pd.NA

In [94]:
# Tidy matrix: genes are either on (1+ counts) or off (0 counts)
metadata_cols = {
    "source_folder", "name", "nucleus",
    "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_centroid_z", 
    "Xmax", "Ymax", "Zmax", "n_endo_markers", "n_fibro_markers",
    "endothelial_score", "fibroblast_score", "cell_type", "confident", "has_reads"
}
# Drop FP readouts
gene_columns = [c for c in all_data.columns if c not in metadata_cols]
fp_columns = [c for c in gene_columns if c.startswith("FP")]
true_gene_columns = [c for c in gene_columns if not c.startswith("FP")]
all_data[true_gene_columns] = (all_data[true_gene_columns].fillna(0).astype(float) > 0).astype(int)
# Create a readable gene state for on genes (e.g. 1_4_12)
all_data["state_code"] = all_data[true_gene_columns].apply(
    lambda row: "_".join(str(i + 1) for i, value in enumerate(row) if value == 1), axis=1,
).replace("", "0")  # all-negative cells get the code "0"
true_gene_dictionary = {c: i for i, c in enumerate(true_gene_columns)}

# # binary_state: one digit per gene in column order, e.g. "100100000001000000"
all_data["binary_state"] = all_data[true_gene_columns].astype(str).agg("".join, axis=1)


In [96]:
# Keep the two kinds of label separate:
#   cell_type                  = read-derived label (Endothelial / Fibroblast / Unknown / Zero Reads); NEVER overwritten
#   assigned_unknown_cell_type = lineage guessed for Zero Reads AND Unknown cells (NA everywhere else)
conf_endo = int(((all_data["cell_type"] == "Endothelial") & (all_data["confident"] == "Confident")).sum())
conf_fibro = int(((all_data["cell_type"] == "Fibroblast") & (all_data["confident"] == "Confident")).sum())
p_fibro = conf_fibro / (conf_endo + conf_fibro)

unassigned_cells = all_data.index[all_data["cell_type"].isin(["Zero Reads", "Unknown"])].to_numpy().copy()
rng = np.random.default_rng(0)
rng.shuffle(unassigned_cells)  # shuffle so the split isn't ordered by index (avoids spatial bias)
n_fibro = int(round(p_fibro * len(unassigned_cells)))
n_endo = len(unassigned_cells) - n_fibro
print(n_fibro, n_endo)

all_data["assigned_unknown_cell_type"] = pd.NA
all_data.loc[unassigned_cells[:n_fibro], "assigned_unknown_cell_type"] = "Fibroblast"
all_data.loc[unassigned_cells[n_fibro:], "assigned_unknown_cell_type"] = "Endothelial"

all_data

319 4569


,source_folder,name,nucleus,CDH5,MMP1,COL1A1,KDR,VWF,PECAM1,VEGFA,...,n_endo_markers,n_fibro_markers,endothelial_score,fibroblast_score,cell_type,confident,has_reads,state_code,binary_state,assigned_unknown_cell_type
0,dev10_1_day2,dev10_1_day2,1,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
1,dev10_1_day2,dev10_1_day2,2,1,0,0,0,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,1,100000000000000000,<NA>
2,dev10_1_day2,dev10_1_day2,3,1,0,0,0,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,1,100000000000000000,<NA>
3,dev10_1_day2,dev10_1_day2,5,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
4,dev10_1_day2,dev10_1_day2,6,0,0,0,1,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,4,000100000000000000,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13042,dev9_4_day4,dev9_4_day4,1418,1,0,0,0,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,1,100000000000000000,<NA>
13043,dev9_4_day4,dev9_4_day4,1423,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
13044,dev9_4_day4,dev9_4_day4,1424,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
13045,dev9_4_day4,dev9_4_day4,1426,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial


In [97]:
# Convert to physical coordinates witin the image (each image has its own coordinate origin)
all_data["x_um"] = all_data["nucleus_centroid_x"] * XY_UM
all_data["y_um"] = all_data["nucleus_centroid_y"] * XY_UM
all_data["z_um"] = all_data["nucleus_centroid_z"] * Z_UM

# Binary gene matrix (cells x genes) reused throughout.
X = all_data[list(true_gene_columns)  ].astype(int).to_numpy()

print("\nCell-type counts:")
print(all_data["cell_type"].value_counts())

print(f"All cells:                     {len(all_data):,}")
print(f"Cells with >=1 transcript:     {all_data['has_reads'].sum():,} "
      f"({100 * all_data['has_reads'].mean():.1f}%)")

print("\nConfident cells per lineage (used for the lineage-specific pairwise):")
print(all_data.loc[all_data["confident"] == "Confident", "cell_type"].value_counts())

print("\nCell type x confidence tiers:")
print(pd.crosstab(all_data["cell_type"], all_data["confident"], dropna=False))


Cell-type counts:
cell_type
Endothelial    7325
Zero Reads     3575
Unknown        1313
Fibroblast      834
Name: count, dtype: int64
All cells:                     13,047
Cells with >=1 transcript:     9,472 (72.6%)

Confident cells per lineage (used for the lineage-specific pairwise):
cell_type
Endothelial    4163
Fibroblast      291
Name: count, dtype: int64

Cell type x confidence tiers:
confident    Confident  Tentative   NaN
cell_type                              
Endothelial       4163       3162     0
Fibroblast         291        543     0
Unknown              0          0  1313
Zero Reads           0          0  3575


In [98]:
all_data.to_csv("clean_data_with_assigned_cell_type.csv")

In [99]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(13047, 18))